# Export tree datasets to Parquet

Four datasets go out as **GeoParquet** (geometry preserved, readable by GeoPandas, DuckDB, QGIS 3.28+,
Apache Sedona):

| name | source | what it is |
|---|---|---|
| `city_public` | `references/Baeume_SFM_2026.gpkg` | Magdeburg tree registry — public green and street trees (85,302 points) |
| `city_private` | `references/Baeume_Liegenschaftsservice_2026.gpkg` | Magdeburg tree registry — municipal-property trees (5,665 points) |
| `city_all` | both of the above | the two registries concatenated, with a `source` column |
| `trees_merged` | `data/orthophotos/segments/250m/ovgu_bbox_tcd_segformer_trees_merged.fgb` | pipeline output for `ovgu_bbox` at the 250 m tile size: crown polygons already enriched with Baumkataster heights and species |

The two registries are **complementary, not overlapping** — no Liegenschaftsservice point lies within 1 m
of an SFM point citywide, so `city_all` double-counts nothing. The verify cell re-checks that rather
than taking it on trust.

`city_all` is built by the pipeline's own `load_baumkataster()` (`src/shadow/cadastre.py`), not by a
concatenation written here, so the exported file is exactly the frame the rest of the project analyses.
That function harmonises the one schema difference between the registries — SFM spells the tree number
`baumnummer`, Liegenschaftsservice spells it `Baumnummer` — and adds `source` with the originating file
stem, so per-registry statistics stay recoverable after the merge.

Only the **250 m** merged crown layer is exported; the 100/500/1000 m variants exist but are alternative
tilings of the same area, and mixing them would double-count crowns.

Everything configurable is in the **Fields** cell — nothing below it needs editing.

## Fields

In [1]:
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "test_notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.shadow.cadastre import load_baumkataster

# --- fields ----------------------------------------------------------------
OUT_DIR     = ROOT / "data" / "parquet"   # where the .parquet files are written
TILE_SIZE_M = 250                         # which merged tiling to export (250 only, see above)
COMPRESSION = "zstd"                      # "snappy" | "zstd" | "gzip" | None
OVERWRITE   = True                        # False = leave existing .parquet files alone
MERGED_NAME = "city_all"                  # output name for public + private combined

SOURCES = {
    "city_public":  ROOT / "references" / "Baeume_SFM_2026.gpkg",
    "city_private": ROOT / "references" / "Baeume_Liegenschaftsservice_2026.gpkg",
    "trees_merged": ROOT / "data" / "orthophotos" / "segments" / f"{TILE_SIZE_M}m"
                         / "ovgu_bbox_tcd_segformer_trees_merged.fgb",
}

# the registries that get concatenated into MERGED_NAME, in this order
REGISTRIES = ["city_public", "city_private"]
# ---------------------------------------------------------------------------

for name, src in SOURCES.items():
    assert src.exists(), f"input missing: {src}"
print(f"{len(SOURCES)} inputs found, writing to {OUT_DIR.relative_to(ROOT)}/")

3 inputs found, writing to data/parquet/


## Convert

`GeoDataFrame.to_parquet` writes GeoParquet: the geometry column is stored as WKB with the CRS carried
in the file metadata, so a round trip needs no reprojection or re-declaration.

In [2]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

def write(name: str, gdf: gpd.GeoDataFrame, src_bytes: int) -> dict | None:
    """Write one GeoParquet file and return a summary row (None if skipped)."""
    dst = OUT_DIR / f"{name}.parquet"
    if dst.exists() and not OVERWRITE:
        print(f"skip   {name}  (exists, OVERWRITE=False)")
        return None
    gdf.to_parquet(dst, compression=COMPRESSION, index=False)
    print(f"wrote  {name}  ->  {dst.relative_to(ROOT)}")
    return {
        "name":       name,
        "rows":       len(gdf),
        "columns":    len(gdf.columns),
        "geom_type":  gdf.geom_type.value_counts().idxmax(),
        "crs":        gdf.crs.to_string(),
        "src_MB":     round(src_bytes / 1e6, 2),
        "parquet_MB": round(dst.stat().st_size / 1e6, 2),
    }

rows = []

# 1 — straight file-to-file conversions
for name, src in SOURCES.items():
    row = write(name, gpd.read_file(src), src.stat().st_size)
    if row:
        rows.append(row)

# 2 — the merged registry, via the pipeline's own loader so the export cannot
#     drift from what src/ analyses
reg_paths = [SOURCES[n] for n in REGISTRIES]
city_all = load_baumkataster(reg_paths)
row = write(MERGED_NAME, city_all, sum(p.stat().st_size for p in reg_paths))
if row:
    rows.append(row)

summary = pd.DataFrame(rows)
summary["ratio"] = (summary["src_MB"] / summary["parquet_MB"]).round(1)
summary

wrote  city_public  ->  data/parquet/city_public.parquet
wrote  city_private  ->  data/parquet/city_private.parquet
wrote  trees_merged  ->  data/parquet/trees_merged.parquet


wrote  city_all  ->  data/parquet/city_all.parquet


,name,rows,columns,geom_type,crs,src_MB,parquet_MB,ratio
0,city_public,85302,9,Point,EPSG:25832,17.06,1.91,8.9
1,city_private,5665,9,Point,EPSG:25832,3.49,0.13,26.8
2,trees_merged,1434,14,Polygon,EPSG:25832,3.07,0.72,4.3
3,city_all,90967,10,Point,EPSG:25832,20.55,2.01,10.2


## Verify the round trip

A conversion that silently drops rows, loses the CRS, or mangles geometry is worse than no conversion,
so read each file back and compare it against its source rather than trusting the write.

In [3]:
for name, src in SOURCES.items():
    a = gpd.read_file(src)
    b = gpd.read_parquet(OUT_DIR / f"{name}.parquet")

    assert len(a) == len(b), f"{name}: row count changed {len(a)} -> {len(b)}"
    assert list(a.columns) == list(b.columns), f"{name}: columns changed"
    assert a.crs == b.crs, f"{name}: CRS changed {a.crs} -> {b.crs}"
    assert a.geometry.geom_equals_exact(b.geometry, tolerance=0).all(), f"{name}: geometry changed"

    non_geom = [c for c in a.columns if c != a.geometry.name]
    pd.testing.assert_frame_equal(a[non_geom], b[non_geom], check_dtype=False)

    print(f"{name:14s} OK   {len(b):6,} rows | {len(b.columns):2d} cols | {b.crs.to_string()}")

city_public    OK   85,302 rows |  9 cols | EPSG:25832
city_private   OK    5,665 rows |  9 cols | EPSG:25832
trees_merged   OK    1,434 rows | 14 cols | EPSG:25832


### …and the merge

`city_all` has no single source file to diff against, so it is checked against its two parts instead:
every row accounted for, the concatenation order intact, the `Baumnummer` capitalisation harmonised,
and — the claim the merge rests on — no point from one registry sitting on top of a point from the
other.

In [4]:
parts = [gpd.read_parquet(OUT_DIR / f"{n}.parquet") for n in REGISTRIES]
merged = gpd.read_parquet(OUT_DIR / f"{MERGED_NAME}.parquet")

# every row is present, exactly once
assert len(merged) == sum(len(p) for p in parts), "merged row count != sum of parts"
expected = {SOURCES[n].stem: len(p) for n, p in zip(REGISTRIES, parts)}
assert merged["source"].value_counts().to_dict() == expected, "source column mislabels rows"

# the tree-number column is harmonised to the lowercase spelling, not duplicated
assert "baumnummer" in merged.columns and "Baumnummer" not in merged.columns

# geometry survives the concatenation, in order
off = 0
for name, part in zip(REGISTRIES, parts):
    got = merged.geometry.iloc[off:off + len(part)].reset_index(drop=True)
    assert got.geom_equals_exact(part.geometry, tolerance=0).all(), f"{name}: geometry reordered"
    assert merged.crs == part.crs, f"{name}: CRS changed"
    off += len(part)

# the registries really are disjoint: nothing within 1 m across the two
a, b = parts
near = gpd.sjoin_nearest(b[["geometry"]], a[["geometry"]], max_distance=1.0, how="inner")
assert len(near) == 0, f"{len(near)} points lie within 1 m across registries — the merge double-counts"

print(f"{MERGED_NAME:14s} OK   {len(merged):6,} rows | {len(merged.columns):2d} cols | "
      f"{merged.crs.to_string()}")
print("               " + "  ".join(f"{k}={v:,}" for k, v in expected.items()))
print("               0 cross-registry points within 1 m")

city_all       OK   90,967 rows | 10 cols | EPSG:25832
               Baeume_SFM_2026=85,302  Baeume_Liegenschaftsservice_2026=5,665
               0 cross-registry points within 1 m


## What is in each file

In [5]:
for name in [*SOURCES, MERGED_NAME]:
    g = gpd.read_parquet(OUT_DIR / f"{name}.parquet")
    print(f"\n=== {name} — {len(g):,} rows ===")
    print(g.dtypes.to_string())


=== city_public — 85,302 rows ===
Gattung lang           object
Stammumfang           float64
Baumhoehe             float64
Kronendurchmesser     float64
Pflanzjahr              int32
Objektbezeichnung      object
Objektart lang         object
baumnummer             object
geometry             geometry

=== city_private — 5,665 rows ===
Gattung lang           object
Baumnummer             object
Stammumfang           float64
Baumhoehe             float64
Kronendurchmesser     float64
Pflanzjahr              int32
Objektbezeichnung      object
Objektart lang         object
geometry             geometry

=== trees_merged — 1,434 rows ===
tree_id                      int32
height_m                   float64
allometric_height_m        float64
crown_radius_m             float64
crown_area_m2              float64
vegetation_model            object
species                     object
height_source               object
is_deciduous                object
trunk_circumference_cm      object
plant


=== city_all — 90,967 rows ===
Gattung lang           object
Stammumfang           float64
Baumhoehe             float64
Kronendurchmesser     float64
Pflanzjahr              int32
Objektbezeichnung      object
Objektart lang         object
baumnummer             object
geometry             geometry
source                 object


## Reading them back

```python
import geopandas as gpd
trees = gpd.read_parquet("data/parquet/trees_merged.parquet")     # full layer, geometry intact
city  = gpd.read_parquet("data/parquet/city_all.parquet")         # both registries, one frame
```

`city_all` keeps `source`, so either registry can be recovered without going back to the GeoPackages:

```python
public = city[city["source"] == "Baeume_SFM_2026"]
```

Parquet is columnar, so a subset of fields can be read without touching the rest of the file — useful
on `city_all`, where most analyses need three columns out of ten:

```python
import pandas as pd
h = pd.read_parquet("data/parquet/city_all.parquet",
                    columns=["Gattung lang", "Baumhoehe", "Kronendurchmesser"])
```

Note `pd.read_parquet` returns a plain DataFrame with no geometry; use `gpd.read_parquet` when the
points or polygons are needed.